In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import importlib.util
import json, math, time, warnings
import subprocess, sys
import numpy as np
import pandas as pd

# Kaggle's GPU image may not include NumPyro. Install only NumPyro so the
# preinstalled CUDA-enabled JAX version is left untouched.
if importlib.util.find_spec('numpyro') is None:
    print('Installing NumPyro 0.22.0 (leaving Kaggle JAX/CUDA packages unchanged)...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps', 'numpyro==0.22.0'], check=True)

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')

try:
    import jax
    import jax.numpy as jnp
    import numpyro
    import numpyro.distributions as dist
    from numpyro.infer import MCMC, NUTS
    from numpyro.infer.initialization import init_to_value
    from numpyro.diagnostics import summary as numpyro_summary
    from astropy.timeseries import BoxLeastSquares
    from scipy.optimize import minimize
    from scipy.special import logsumexp
except ImportError as exc:
    raise RuntimeError(f'Missing runtime dependency: {exc}. Use a Kaggle image with JAX, NumPyro, Astropy, and SciPy.') from exc

print('JAX backend:', jax.default_backend())
print('JAX devices:', jax.devices())
print('NumPyro:', numpyro.__version__)

# Find sibling binary and white_dwarf simulation folders from a Day 7 output.
roots = {}
for meta_path in INPUT.rglob('simulation_metadata.json'):
    if meta_path.parent.name not in {'binary', 'white_dwarf'}:
        continue
    try:
        meta = json.loads(meta_path.read_text())
    except Exception as exc:
        print('Skipping unreadable metadata:', meta_path, exc)
        continue
    family = meta.get('family')
    if family not in {'binary', 'white_dwarf'}:
        continue
    root = meta_path.parent.parent
    roots.setdefault(root, {})[family] = (meta_path.parent, meta)

usable = []
for root, entries in roots.items():
    if {'binary', 'white_dwarf'}.issubset(entries):
        if all(entries[f][1].get('parameter_draws_unique', 0) >= 100_000 for f in ('binary', 'white_dwarf')):
            created = max(entries[f][1].get('created_utc', '') for f in ('binary', 'white_dwarf'))
            usable.append((created, root, entries))
if not usable:
    raise FileNotFoundError(
        'Could not find one attached Day 7 simulation root containing both binary and white_dwarf '
        'folders with at least 100,000 unique draws each. Attach the Day 7 Cosmic swan Bayesian '
        'output that contains simulations/{binary,white_dwarf}/observations.npy and parameters.csv.'
    )
_, SIM_ROOT, SIMS = sorted(usable, key=lambda item: item[0])[-1]
BINARY_DIR, BINARY_META = SIMS['binary']
WD_DIR, WD_META = SIMS['white_dwarf']

for family, folder in [('binary', BINARY_DIR), ('white_dwarf', WD_DIR)]:
    for filename in ('observations.npy', 'parameters.csv', 'simulation_metadata.json'):
        if not (folder / filename).is_file():
            raise FileNotFoundError(f'Missing {family} simulation artifact: {folder / filename}')

binary_obs = np.load(BINARY_DIR / 'observations.npy', mmap_mode='r')
binary_params = pd.read_csv(BINARY_DIR / 'parameters.csv')
time_days = np.load(BINARY_DIR / 'time_days.npy')
wd_obs = np.load(WD_DIR / 'observations.npy', mmap_mode='r')
wd_params = pd.read_csv(WD_DIR / 'parameters.csv')

assert len(binary_params) == binary_obs.shape[0] >= 100_000
assert len(wd_params) == wd_obs.shape[0] >= 100_000
assert {'period_days','phase','depth_primary','depth_ratio','duration_fraction','ellipsoidal_amplitude','reflection_amplitude','noise_sigma'}.issubset(binary_params.columns)
assert {'mass_solar','radius_error_solar'}.issubset(wd_params.columns)
assert binary_obs.shape[1] == len(time_days)

OUT = WORK / ('day9-binary-wd-validation-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
OUT.mkdir(parents=True, exist_ok=False)
print('[PASS] Day 7 simulation root:', SIM_ROOT)
print('[PASS] Binary observations/parameters:', binary_obs.shape, len(binary_params))
print('[PASS] White-dwarf observations/parameters:', wd_obs.shape, len(wd_params))
print('[PASS] Output directory:', OUT)
print('Preflight passed. Sampling has not started.')

In [ ]:
from scipy.special import logsumexp

N_WD = 500
WD_SEED = 90421
MASS_GRID = np.linspace(0.15, 1.35, 1601)
SCATTER_GRID = np.linspace(0.0, 0.01, 101)
SCATTER_SCALE = 0.002

rng_wd = np.random.default_rng(WD_SEED)
wd_indices = np.sort(rng_wd.choice(len(wd_params), size=min(N_WD, len(wd_params)), replace=False))
wd_truth = wd_params.iloc[wd_indices]['mass_solar'].to_numpy(float)
wd_error = np.asarray(wd_obs[wd_indices, 1], dtype=float)
wd_radius = np.asarray(wd_obs[wd_indices, 0], dtype=float)

m_ch = 1.44
term = (m_ch / MASS_GRID) ** (2.0 / 3.0) - (MASS_GRID / m_ch) ** (2.0 / 3.0)
radius_grid = 0.0112 * np.sqrt(np.maximum(term, 1e-8))
ds = SCATTER_GRID[1] - SCATTER_GRID[0]
scatter_weights = np.ones_like(SCATTER_GRID) * ds
scatter_weights[[0, -1]] *= 0.5
log_scatter_weights = (
    np.log(scatter_weights)
    + np.log(np.sqrt(2.0 / np.pi) / SCATTER_SCALE)
    - 0.5 * (SCATTER_GRID / SCATTER_SCALE) ** 2
)

wd_rows = []
for local_i, (idx, observed, err, truth) in enumerate(zip(wd_indices, wd_radius, wd_error, wd_truth), start=1):
    # Matched baseline: the simulator has measurement noise and no extra scatter.
    variance = err**2
    matched_logp = -0.5 * ((observed-radius_grid)**2 / variance + np.log(2.0*np.pi*variance))
    matched_logp -= logsumexp(matched_logp)
    matched_prob = np.exp(matched_logp)
    matched_prob /= matched_prob.sum()
    matched_cdf = np.cumsum(matched_prob)
    matched_q = np.interp([0.05, 0.25, 0.50, 0.75, 0.95], matched_cdf, MASS_GRID)
    wd_rows.append({
        'simulation_row': int(idx), 'inference_model': 'matched_no_intrinsic_scatter',
        'known_mass_solar': float(truth), 'posterior_q05': float(matched_q[0]),
        'posterior_q25': float(matched_q[1]), 'posterior_median': float(matched_q[2]),
        'posterior_q75': float(matched_q[3]), 'posterior_q95': float(matched_q[4]),
        'covered_50pct': bool(matched_q[1] <= truth <= matched_q[3]),
        'covered_90pct': bool(matched_q[0] <= truth <= matched_q[4]),
        'measurement_error_solar': float(err),
    })

    # Project-model fit: marginalize over the inference model's HalfNormal(0.002 R_sun) scatter prior.
    variance_scatter = err**2 + SCATTER_GRID[None, :]**2
    residual = observed - radius_grid[:, None]
    log_likelihood = -0.5 * (residual**2 / variance_scatter + np.log(2.0*np.pi*variance_scatter))
    model_logp = logsumexp(log_likelihood + log_scatter_weights[None, :], axis=1)
    model_logp -= logsumexp(model_logp)
    model_prob = np.exp(model_logp)
    model_prob /= model_prob.sum()
    model_cdf = np.cumsum(model_prob)
    model_q = np.interp([0.05, 0.25, 0.50, 0.75, 0.95], model_cdf, MASS_GRID)
    wd_rows.append({
        'simulation_row': int(idx), 'inference_model': 'project_model_with_intrinsic_scatter',
        'known_mass_solar': float(truth), 'posterior_q05': float(model_q[0]),
        'posterior_q25': float(model_q[1]), 'posterior_median': float(model_q[2]),
        'posterior_q75': float(model_q[3]), 'posterior_q95': float(model_q[4]),
        'covered_50pct': bool(model_q[1] <= truth <= model_q[3]),
        'covered_90pct': bool(model_q[0] <= truth <= model_q[4]),
        'measurement_error_solar': float(err),
    })
    if local_i % 100 == 0:
        print(f'White-dwarf posterior cases: {local_i}/{len(wd_indices)} (each fit under both scatter assumptions)')

wd_results = pd.DataFrame(wd_rows)
wd_results.to_csv(OUT / 'day9_wd_recovery_cases.csv', index=False)

def summarize_wd_group(group):
    errors = group['posterior_median'].to_numpy() - group['known_mass_solar'].to_numpy()
    n = len(group)
    covered = int(group['covered_90pct'].sum())
    p = covered / n
    z = 1.96
    den = 1 + z*z/n
    wilson_center = (p + z*z/(2*n)) / den
    wilson_half = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / den
    return {
        'n_cases': int(n),
        'median_bias_solar': float(np.median(errors)),
        'mean_bias_solar': float(np.mean(errors)),
        'mae_solar': float(np.mean(np.abs(errors))),
        'rmse_solar': float(np.sqrt(np.mean(errors**2))),
        'coverage_50pct': float(group['covered_50pct'].mean()),
        'coverage_90pct': float(p),
        'coverage_90pct_wilson_95pct_interval': [float(wilson_center-wilson_half), float(wilson_center+wilson_half)],
        'median_90pct_interval_width_solar': float((group['posterior_q95']-group['posterior_q05']).median()),
    }

wd_summary = {
    method: summarize_wd_group(group)
    for method, group in wd_results.groupby('inference_model', sort=True)
}
wd_summary['interpretation'] = {
    'matched_no_intrinsic_scatter': 'Matches the simulator exactly: mass-radius relation plus the recorded radius measurement error.',
    'project_model_with_intrinsic_scatter': 'Uses the project inference model with a HalfNormal(0.002 R_sun) intrinsic-scatter nuisance prior; the simulator does not generate this extra scatter.',
    'coverage_note': 'These are synthetic prior-predictive cases. Compare interval coverage and width between both assumptions; do not present one realization as proof of universal calibration.',
}
(OUT / 'day9_wd_recovery_summary.json').write_text(json.dumps(wd_summary, indent=2))
print(json.dumps(wd_summary, indent=2))

In [ ]:
from scipy.optimize import minimize

N_BINARY_CASES = 4
BINARY_SEED = 91027
WARMUP = 300
SAMPLES = 400
CHAINS = 2
SAMPLE_INTERVAL = float(np.median(np.diff(time_days)))
BASELINE = float(np.ptp(time_days))

# Predeclared observability criteria, computed from simulation truth only to define
# the recoverable subset; all selected cases are still evaluated, including failures.
periods = binary_params['period_days'].to_numpy(float)
phases = binary_params['phase'].to_numpy(float)
depths = binary_params['depth_primary'].to_numpy(float)
ratios = binary_params['depth_ratio'].to_numpy(float)
durations = binary_params['duration_fraction'].to_numpy(float)
noise_truths = binary_params['noise_sigma'].to_numpy(float)
event_samples = periods * durations / SAMPLE_INTERVAL
eclipses_over_baseline = BASELINE * durations / SAMPLE_INTERVAL
period_samples = periods / SAMPLE_INTERVAL
snr_proxy = depths / noise_truths
eligible = (event_samples >= 2.0) & (eclipses_over_baseline >= 5.0) & (period_samples >= 2.0) & (snr_proxy >= 5.0)
eligible_indices = np.flatnonzero(eligible)
if len(eligible_indices) < N_BINARY_CASES:
    raise RuntimeError(f'Only {len(eligible_indices)} simulations meet the declared binary observability rules.')
rng_binary = np.random.default_rng(BINARY_SEED)
case_indices = np.sort(rng_binary.choice(eligible_indices, size=N_BINARY_CASES, replace=False))
print(f'Eligible binary simulations: {len(eligible_indices)} / {len(binary_params)}')
print('Selected simulation rows:', case_indices.tolist())
print(f'Binary NUTS configuration: {CHAINS} chains x ({WARMUP} warmup + {SAMPLES} samples) per case')

# Nearly exact smooth approximation to the simulator clip, preserving gradients.
def smooth_clip(x, sharpness=500.0):
    return (jax.nn.softplus(sharpness*x) - jax.nn.softplus(sharpness*(x-1.0))) / sharpness

def binary_prediction(time_j, period, phase, depth, ratio, duration, ellip, reflection):
    orbital = jnp.mod((time_j / period) - phase + 0.5, 1.0) - 0.5
    secondary_phase = jnp.mod(orbital, 1.0) - 0.5
    half_width = jnp.maximum(duration / 2.0, 1e-6)
    ingress = jnp.maximum(half_width * 0.18, 1e-6)
    primary = smooth_clip((half_width + ingress - jnp.abs(orbital)) / (2.0 * ingress))
    secondary = smooth_clip((half_width + ingress - jnp.abs(secondary_phase)) / (2.0 * ingress))
    return (1.0 - depth*primary - depth*ratio*secondary
            + ellip*jnp.cos(4.0*jnp.pi*orbital)
            + reflection*jnp.cos(2.0*jnp.pi*orbital))

def get_bls_start(t, y):
    # Search period bands separately. Astropy requires the longest trial
    # duration to be shorter than the band's minimum period.
    bls = BoxLeastSquares(t, y)
    max_period = min(13.0, np.ptp(t) / 2.0)
    period_bands = [(0.1, 0.2), (0.2, 0.4), (0.4, 0.8), (0.8, 1.6),
                    (1.6, 3.2), (3.2, 6.4), (6.4, 13.0)]
    trials = []
    for period_low, period_high in period_bands:
        period_high = min(period_high, max_period)
        if period_high <= period_low:
            continue
        max_duration = min(1.5, 0.95 * period_low, 0.12 * period_high)
        min_duration = min(0.005, max_duration / 2.0)
        durations_days = np.geomspace(min_duration, max_duration, 16)
        result = bls.autopower(duration=durations_days, objective='snr',
                               minimum_period=period_low, maximum_period=period_high,
                               frequency_factor=12.0)
        best = int(np.nanargmax(result.power))
        trials.append((float(result.power[best]), float(result.period[best]),
                       float(result.transit_time[best]), float(result.depth[best]),
                       float(result.duration[best])))
    if not trials:
        raise ValueError('BLS produced no period trials')
    _, period, transit_time, depth, duration_days = max(trials, key=lambda item: item[0])
    return {
        'period': period,
        'phase': float(np.mod(transit_time / period, 1.0)),
        'depth': float(np.clip(depth, 0.002, 0.45)),
        'duration': float(np.clip(duration_days / period, 0.005, 0.12)),
    }

def fit_one_binary(case_index, seed):
    y = np.asarray(binary_obs[case_index], dtype=np.float64)
    t = np.asarray(time_days, dtype=np.float64)
    start = get_bls_start(t, y)
    period_low = max(0.1, start['period'] - 0.04)
    period_high = min(13.0, start['period'] + 0.04)
    if period_high <= period_low:
        raise ValueError('BLS period has no valid local period window')
    phase0 = start['phase']
    time_j, observed_j = jnp.asarray(t), jnp.asarray(y)
    log_noise_low, log_noise_high = np.log(1e-4), np.log(0.03)

    def predict_from_params(theta):
        period, phase_delta, depth, ratio, duration, ellip, reflection, log_noise = theta
        phase = jnp.mod(phase0 + phase_delta, 1.0)
        return binary_prediction(time_j, period, phase, depth, ratio, duration, ellip, reflection), jnp.exp(log_noise)

    # Use observed residual scatter for an initial noise scale; do not use the known truth.
    rough_noise = np.std(np.diff(y)) / np.sqrt(2.0)
    rough_noise = float(np.clip(rough_noise, 1e-4, 0.03))
    bounds = [(period_low, period_high), (-0.08, 0.08), (0.002, 0.45), (0.0, 1.0),
              (0.005, 0.12), (0.0, 0.02), (-0.01, 0.01), (log_noise_low, log_noise_high)]
    initial = np.array([start['period'], 0.0, start['depth'], 0.5, start['duration'], 0.005, 0.0, np.log(rough_noise)])
    initial = np.array([np.clip(value, lo + 1e-8, hi - 1e-8) for value, (lo, hi) in zip(initial, bounds)])

    def objective(theta):
        prediction, sigma = predict_from_params(jnp.asarray(theta))
        residual = observed_j - prediction
        return float(0.5*jnp.sum((residual/sigma)**2 + 2.0*jnp.log(sigma) + np.log(2.0*np.pi)))

    map_result = minimize(objective, initial, method='L-BFGS-B', bounds=bounds,
                          options={'maxiter': 1200, 'ftol': 1e-10})
    if not np.isfinite(map_result.fun):
        raise RuntimeError('Binary MAP optimization returned a non-finite objective')
    map_theta = np.asarray(map_result.x, dtype=float)
    init_values = {
        'period_days': float(map_theta[0]), 'phase_offset': float(map_theta[1]),
        'depth_primary': float(map_theta[2]), 'depth_ratio': float(map_theta[3]),
        'duration_fraction': float(map_theta[4]), 'ellipsoidal_amplitude': float(map_theta[5]),
        'reflection_amplitude': float(map_theta[6]), 'log_noise': float(map_theta[7]),
    }

    def model():
        period = numpyro.sample('period_days', dist.Uniform(period_low, period_high))
        phase_offset = numpyro.sample('phase_offset', dist.Uniform(-0.08, 0.08))
        depth = numpyro.sample('depth_primary', dist.Uniform(0.002, 0.45))
        ratio = numpyro.sample('depth_ratio', dist.Uniform(0.0, 1.0))
        duration = numpyro.sample('duration_fraction', dist.Uniform(0.005, 0.12))
        ellip = numpyro.sample('ellipsoidal_amplitude', dist.Uniform(0.0, 0.02))
        reflection = numpyro.sample('reflection_amplitude', dist.Uniform(-0.01, 0.01))
        log_noise = numpyro.sample('log_noise', dist.Uniform(log_noise_low, log_noise_high))
        phase = jnp.mod(phase0 + phase_offset, 1.0)
        prediction = binary_prediction(time_j, period, phase, depth, ratio, duration, ellip, reflection)
        numpyro.sample('flux', dist.Normal(prediction, jnp.exp(log_noise)), obs=observed_j)

    kernel = NUTS(model, init_strategy=init_to_value(values=init_values), target_accept_prob=0.92, max_tree_depth=9)
    sampler = MCMC(kernel, num_warmup=WARMUP, num_samples=SAMPLES, num_chains=CHAINS,
                   chain_method='parallel', progress_bar=True)
    sampler.run(jax.random.PRNGKey(seed))
    grouped = sampler.get_samples(group_by_chain=True)
    stats = numpyro_summary(grouped, group_by_chain=True)
    finite_rhat = [float(np.asarray(v['r_hat'])) for v in stats.values() if np.isfinite(np.asarray(v['r_hat'])).all()]
    finite_ess = [float(np.asarray(v['n_eff'])) for v in stats.values() if np.isfinite(np.asarray(v['n_eff'])).all()]
    divergences = int(np.asarray(sampler.get_extra_fields(group_by_chain=True)['diverging']).sum())
    max_rhat = max(finite_rhat) if finite_rhat else float('nan')
    min_ess = min(finite_ess) if finite_ess else float('nan')
    status = 'screen_pass' if max_rhat <= 1.05 and min_ess >= 100 and divergences == 0 else 'needs_review'
    flat = sampler.get_samples(group_by_chain=False)
    truth_row = binary_params.iloc[case_index]
    truth_values = {
        'period_days': float(truth_row['period_days']),
        'phase_offset': float(((float(truth_row['phase']) - phase0 + 0.5) % 1.0) - 0.5),
        'depth_primary': float(truth_row['depth_primary']),
        'depth_ratio': float(truth_row['depth_ratio']),
        'duration_fraction': float(truth_row['duration_fraction']),
        'ellipsoidal_amplitude': float(truth_row['ellipsoidal_amplitude']),
        'reflection_amplitude': float(truth_row['reflection_amplitude']),
        'log_noise': float(np.log(float(truth_row['noise_sigma']))),
    }
    case_rows = []
    for name, values in flat.items():
        arr = np.asarray(values, dtype=float)
        q05, median, q95 = np.quantile(arr, [0.05, 0.50, 0.95])
        truth = truth_values[name]
        case_rows.append({
            'simulation_row': int(case_index), 'parameter': name,
            'q05': float(q05), 'median': float(median), 'q95': float(q95),
            'known_truth': float(truth), 'truth_inside_90pct_interval': bool(q05 <= truth <= q95),
        })
    # Report circular phase as the BLS-relative offset; this avoids false wrap errors.
    result = {
        'simulation_row': int(case_index), 'status': status,
        'bls_period_days': start['period'], 'period_window_days': [period_low, period_high],
        'known_period_days': float(truth_row['period_days']),
        'known_period_inside_local_window': bool(period_low <= float(truth_row['period_days']) <= period_high),
        'map_success': bool(map_result.success), 'map_message': str(map_result.message),
        'max_r_hat': max_rhat, 'min_effective_sample_size': min_ess,
        'divergences': divergences,
        'screening_thresholds': {'max_r_hat': 1.05, 'min_effective_sample_size': 100, 'divergences': 0},
        'n_observations': int(len(y)), 'seed': int(seed),
    }
    samples_to_save = {name: np.asarray(value) for name, value in grouped.items()}
    np.savez_compressed(OUT / f'binary_case_{case_index}_posterior.npz', **samples_to_save)
    return result, case_rows

binary_summaries = []
binary_interval_rows = []
for case_num, index in enumerate(case_indices, start=1):
    print(f'\nBinary case {case_num}/{len(case_indices)} (simulation row {index})')
    start_time = time.time()
    try:
        result, intervals = fit_one_binary(int(index), BINARY_SEED + case_num)
        result['seconds'] = round(time.time() - start_time, 1)
        binary_summaries.append(result)
        binary_interval_rows.extend(intervals)
        print(f"Status={result['status']} | max R-hat={result['max_r_hat']:.4g} | min ESS={result['min_effective_sample_size']:.4g} | divergences={result['divergences']} | {result['seconds']} sec")
    except Exception as exc:
        binary_summaries.append({'simulation_row': int(index), 'status': 'failed', 'error': repr(exc), 'seconds': round(time.time()-start_time, 1)})
        print('Case failed; continuing:', repr(exc))

binary_summary_df = pd.DataFrame(binary_summaries)
binary_intervals_df = pd.DataFrame(binary_interval_rows)
binary_summary_df.to_csv(OUT / 'day9_binary_case_diagnostics.csv', index=False)
binary_intervals_df.to_csv(OUT / 'day9_binary_recovery_intervals.csv', index=False)
print('\nBinary case statuses:')
print(binary_summary_df['status'].value_counts(dropna=False))
if len(binary_intervals_df):
    print('\nDescriptive parameter interval coverage (small n; not calibration):')
    print(binary_intervals_df.groupby('parameter')['truth_inside_90pct_interval'].agg(['sum','count','mean']))

In [ ]:
created = datetime.now(timezone.utc).isoformat()
valid_binary = binary_summary_df[binary_summary_df['status'] == 'screen_pass'] if len(binary_summary_df) else pd.DataFrame()
summary = {
    'created_utc': created,
    'simulation_root': str(SIM_ROOT),
    'jax_backend': jax.default_backend(),
    'jax_devices': [str(device) for device in jax.devices()],
    'scope': 'Synthetic recovery only. White-dwarf exact-grid posterior on a random sample; binary NUTS conditional on local BLS period windows for four resolution/SNR-screened cases.',
    'white_dwarf': wd_summary,
    'binary': {
        'n_cases_selected': int(len(case_indices)),
        'n_cases_screen_pass': int(len(valid_binary)),
        'n_cases_needing_review_or_failed': int(len(binary_summary_df) - len(valid_binary)),
        'selection_criteria': 'at least 2 samples per eclipse, at least 5 primary-eclipse samples over baseline, at least 2 samples per period, and depth/noise_sigma >= 5',
        'conditional_period_window_half_width_days': 0.04,
        'warmup_per_chain': WARMUP, 'samples_per_chain': SAMPLES, 'chains': CHAINS,
        'descriptive_90pct_interval_coverage_by_parameter': (
            binary_intervals_df.groupby('parameter')['truth_inside_90pct_interval'].mean().to_dict()
            if len(binary_intervals_df) else {}
        ),
        'interpretation_limit': 'Only four cases; conditional local-period fits do not validate global alias recovery or calibrated population coverage.',
    },
    'outputs': [
        'day9_wd_recovery_cases.csv', 'day9_wd_recovery_summary.json',
        'day9_binary_case_diagnostics.csv', 'day9_binary_recovery_intervals.csv',
        'binary_case_*_posterior.npz',
    ],
}
(OUT / 'day9_repeated_recovery_summary.json').write_text(json.dumps(summary, indent=2))
print('Saved outputs under:', OUT)
for path in sorted(OUT.iterdir()):
    print(f'{path.name}: {path.stat().st_size / (1024**2):.2f} MiB')
print('\nRun summary:')
print(json.dumps(summary, indent=2))